# F2 multi-depth whitespace v0.2

Fixed F2 geometry; separate global breathing space from near/middle/far omission. Compare old C, terrain with new reserves, and three vegetation grammars. No new projection, source DEM changes, weather model, pigment layer or machine commands.

Install `python -m pip install -e ".[research,dev]"` from the repository root; select that Python environment in VSCode/Jupyter. Keep the original TIFFs outside Git. Set `DATASCAPE_DATA_DIR` to their folder.

In [ ]:
from pathlib import Path
import os, json
import numpy as np
from IPython.display import display, Image
from datascape.rendering.multidepth_whitespace import run

ROOT = Path.cwd()
if not (ROOT / "configs").exists():
    ROOT = ROOT.parent
assert (ROOT / "configs/scenes/seoul_f2_multidepth_whitespace_v02.json").is_file()
DATA = Path(os.environ.get("DATASCAPE_DATA_DIR", ROOT / "data/raw"))
MASTER = DATA / "SeoulNorth_v02_MASTER_30m.tif"
VEG = DATA / "SeoulNorth_v02_VEG_10m.tif"
assert MASTER.is_file() and VEG.is_file()
CONFIG = ROOT / "configs/scenes/seoul_f2_multidepth_whitespace_v02.json"
OUTPUT = ROOT / "outputs/f2_multidepth_whitespace_v02"


In [ ]:
result = run(MASTER, VEG, CONFIG, OUTPUT, repo_root=ROOT)
display(Image(filename=str(OUTPUT / "comparison.png")))
display(Image(filename=str(OUTPUT / "ablation.png")))

## Global vs local

Global top/bottom fields are composition controls. The unchanged F2 already has 24.33% full-width open sky; a 10–20% top envelope cannot shrink existing paper while geometry and placement are fixed. The bottom envelope varies across source features and preserves four measured low-ground patches. Anchors are legibility selections, not observed rocks or roads.

Local fields use source-distance roles and relative terrain features, not screen-height bands. A farther side of a depth discontinuity and a local relief fraction help identify ridge feet. Major ridges and the upper silhouette remain protected.

In [ ]:
for name in ["global_envelope.png", "local_fields.png", "envelope_profiles.png", "gradation.png"]:
    display(Image(filename=str(OUTPUT / name)))

## Depth-aware vegetation

Near marks: trunks, branches, crown arcs. Middle marks: shared clumps and repeated short lines. Far marks: sparse leaf dots/dashes. Roots require measured seasonal NDVI and valid observation/terrain support. These are symbolic previews, not species or reconstructed tree volumes.

Each glyph carries its source root's forward distance. A foreground branch can remain over a local reserve belonging to deeper terrain; nearer terrain occludes a background glyph. Global reserves apply at every depth. Local zero terrain contribution can therefore contain an intentionally visible foreground mark; it is not the same as an unconditional global paper reserve.

In [ ]:
for name in ["vegetation_fields.png", "vegetation_grammar.png", "physical_pair.png"]:
    display(Image(filename=str(OUTPUT / name)))
print(json.dumps(result["record"]["metrics"], indent=2))

In [ ]:
r = result["reserves"]
occupied = result["base"]["occupied"]
local_core = occupied & (r["local_core_retention"] == 0)
global_core = occupied & (r["global_retention"] == 0)
assert result["record"]["metrics"]["baseline_pixels_identical"]
assert result["record"]["metrics"]["previous_artifacts_unchanged"]
assert local_core.any() and global_core.any()
assert np.all(result["terrain_ink"][local_core] == 0)
assert np.all(result["mark_ink"][global_core] == 0)
assert result["record"]["source_inputs"]["master"]["unchanged_after"]
assert result["record"]["source_inputs"]["vegetation"]["unchanged_after"]
print("Verified immutable baseline/input files and depth-scoped zero contribution.")

## Controlled changes

Copy the JSON to a new configuration and write to a new output directory. Tune a single local score transition, reserve extent, anchor width or vegetation role density at a time. Keep the F2 source JSON and v0.1 files fixed. Native noise and mark placement/geometry use separate recorded seed streams. Float16 scalar field archives are diagnostic; boolean core masks preserve exact-zero semantics, and original source XYZ/depth remain in the unchanged v0.1 archive.

This notebook validates composition and omission before any final stroke, color or G-code stage.